# 04. 탐지 데이터셋과 기준 모델 평가

## 연구 목적
수신기 관측값을 시간 window로 구성하고, 물리 기반 통계량과 간단한 기준 모델부터 평가합니다.

## 입력
- `artifacts/datasets/*.csv` 또는 `*.parquet`
- 필수 개념 열: run/scenario ID, time/window, label, PRN별 Doppler/CN₀/추적 특징

처음부터 복잡한 DL을 고정하지 않고 데이터 분리와 baseline을 먼저 확인합니다.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("저장소 루트 또는 notebooks/에서 Notebook을 실행하세요.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
ARTIFACTS = PROJECT_ROOT / "artifacts"
print("PROJECT_ROOT:", PROJECT_ROOT)


## 중간 확인 1 — 데이터셋 준비 상태와 schema

In [ ]:
from gnss_doppler_lab.research_sequence import sequence_status

status = sequence_status(ARTIFACTS)['04_detection_dataset']
print('탐지 데이터셋 준비:', status['ready'])
print('경로:', status['path'] or '아직 없음 — 02/03단계 검증 후 생성')

## 중간 확인 2 — label 분포와 수치 특징 분포

In [ ]:
import csv
from collections import Counter
import matplotlib.pyplot as plt

if not status['ready'] or not str(status['path']).endswith('.csv'):
    print('CSV 데이터셋이 준비되면 label 분포와 첫 수치 특징 histogram을 표시합니다.')
else:
    path = Path(status['path']); rows = list(csv.DictReader(path.open()))
    if not rows or 'label' not in rows[0]:
        raise ValueError('데이터셋에 label 열이 필요합니다.')
    print('rows:', len(rows), 'labels:', Counter(r['label'] for r in rows))
    excluded = {'label','run_id','scenario_id','time','window_id'}
    numeric = []
    for key in rows[0]:
        if key in excluded: continue
        try: [float(r[key]) for r in rows]; numeric.append(key)
        except (ValueError, TypeError): pass
    if numeric:
        feature = numeric[0]
        for label in sorted({r['label'] for r in rows}):
            vals=[float(r[feature]) for r in rows if r['label']==label]
            plt.hist(vals,bins=40,alpha=.5,label=label)
        plt.title(feature); plt.legend(); plt.show()
    else:
        print('표시 가능한 수치 feature가 없습니다.')

## 판정

- [ ] random row split이 아니라 run/scenario 단위로 train/validation/test를 분리한다.
- [ ] power·파일명·공격 시작 위치 같은 label leakage를 검사한다.
- [ ] 평균/분산/공통모드 residual 등 단순 통계 baseline을 먼저 계산한다.
- [ ] ROC-AUC뿐 아니라 PR-AUC, F1, false alarm, detection delay를 보고한다.
- [ ] TEXBAT/OAKBAT/FGI처럼 별도 출처 데이터로 외부 일반화를 평가한다.

## 다음 단계
baseline과 데이터 독립성이 확인된 후에만 시계열 ML/DL 모델을 추가하고, 최종 논문 그림과 표를 고정합니다.